# (임시) 실습 파일 전체 실행 점검
「AI와 데이터 분석」 Colab 실습 파일 35개를 **이 Colab 환경에서 하나씩 처음부터 끝까지 실행**해 오류가 있는지 확인합니다.

**사용법**: 메뉴 **런타임 → 모두 실행** 한 번. 끝나면 맨 아래 표에 결과가 나옵니다(10분 안팎).

확인이 끝나면 이 파일은 저장소에서 삭제합니다. 학생용 자료가 아닙니다.

In [ ]:
import sys, os, time, json, subprocess, urllib.request
import pandas as pd, sklearn, matplotlib
print('Python', sys.version.split()[0], '| pandas', pd.__version__, '| scikit-learn', sklearn.__version__, '| matplotlib', matplotlib.__version__)
RAW = 'https://raw.githubusercontent.com/Daeil-Noh/Deu_Edu_AX_MD/main/'
NOTEBOOKS = ["ai-data-analysis/week02/lesson02.ipynb", "ai-data-analysis/week03/lesson01.ipynb", "ai-data-analysis/week03/lesson02.ipynb", "ai-data-analysis/week03/lesson03.ipynb", "ai-data-analysis/week04/lesson01.ipynb", "ai-data-analysis/week04/lesson02.ipynb", "ai-data-analysis/week04/lesson03.ipynb", "ai-data-analysis/week05/lesson01.ipynb", "ai-data-analysis/week05/lesson02.ipynb", "ai-data-analysis/week05/lesson03.ipynb", "ai-data-analysis/week06/lesson01.ipynb", "ai-data-analysis/week06/lesson02.ipynb", "ai-data-analysis/week06/lesson03.ipynb", "ai-data-analysis/week07/assignment1.ipynb", "ai-data-analysis/week07/lesson01.ipynb", "ai-data-analysis/week07/lesson02.ipynb", "ai-data-analysis/week07/lesson03.ipynb", "ai-data-analysis/week09/lesson01.ipynb", "ai-data-analysis/week09/lesson02.ipynb", "ai-data-analysis/week09/topic_form.ipynb", "ai-data-analysis/week10/lesson01.ipynb", "ai-data-analysis/week10/lesson02.ipynb", "ai-data-analysis/week10/lesson03.ipynb", "ai-data-analysis/week11/assignment2.ipynb", "ai-data-analysis/week11/lesson01.ipynb", "ai-data-analysis/week11/lesson02.ipynb", "ai-data-analysis/week11/lesson03.ipynb", "ai-data-analysis/week12/lesson01.ipynb", "ai-data-analysis/week12/lesson02.ipynb", "ai-data-analysis/week12/lesson03.ipynb", "ai-data-analysis/week13/lesson01.ipynb", "ai-data-analysis/week13/lesson02.ipynb", "ai-data-analysis/week13/lesson03.ipynb", "ai-data-analysis/week14/lesson01.ipynb", "ai-data-analysis/week14/report_template.ipynb"]
os.makedirs('/content/_nbtest', exist_ok=True)
print(len(NOTEBOOKS), '개 파일')

In [ ]:
results = []
for i, path in enumerate(NOTEBOOKS, 1):
    t0 = time.time()
    name = path.replace('ai-data-analysis/', '').replace('/', '_')
    src, out = f'/content/_nbtest/{name}', f'/content/_nbtest/out_{name}'
    row = {'파일': path.replace('ai-data-analysis/', ''), '결과': '', '오류 셀': '', '오류 내용': '', '경고': 0, '시간(초)': 0}
    try:
        urllib.request.urlretrieve(RAW + path, src)
        # 파일마다 새 커널로 처음부터 끝까지 실행 (오류가 나도 끝까지 진행해 결과 파일을 남김)
        p = subprocess.run(['jupyter', 'nbconvert', '--to', 'notebook', '--execute', '--allow-errors',
                            '--ExecutePreprocessor.timeout=600', '--ExecutePreprocessor.kernel_name=python3',
                            '--output-dir', '/content/_nbtest', '--output', f'out_{name}', src], capture_output=True, text=True)
        if not os.path.exists(out):
            raise RuntimeError(p.stderr.strip().splitlines()[-1] if p.stderr.strip() else 'nbconvert 실패')
        nb = json.load(open(out, encoding='utf-8'))
        errs = []
        for k, c in enumerate(c for c in nb['cells'] if c['cell_type'] == 'code'):
            for o in c.get('outputs', []):
                if o.get('output_type') == 'error':
                    errs.append(f"코드 셀 {k + 1}: {o.get('ename')}: {o.get('evalue', '')[:200]}")
                if o.get('name') == 'stderr' and 'Warning' in ''.join(o.get('text', '')):
                    row['경고'] += 1
        row['결과'] = '오류' if errs else 'OK'
        if errs:
            row['오류 셀'] = errs[0].split(':')[0]
            row['오류 내용'] = ' / '.join(errs)
    except Exception as e:
        row['결과'] = '실행 실패'
        row['오류 내용'] = str(e)[:300]
    row['시간(초)'] = round(time.time() - t0, 1)
    results.append(row)
    print(f"[{i:2d}/{len(NOTEBOOKS)}] {row['결과']}  {row['파일']} ({row['시간(초)']}초)" + (f"  ← {row['오류 내용'][:120]}" if row['결과'] != 'OK' else ''))

## 결과

In [ ]:
df = pd.DataFrame(results)
bad = df[df['결과'] != 'OK']
print(f"전체 {len(df)}개 · OK {len(df) - len(bad)}개 · 오류 {len(bad)}개 · 경고가 나온 파일 {(df['경고 수'] > 0).sum()}개")
pd.set_option('display.max_colwidth', 300)
display(bad if len(bad) else df)
if (df['경고'] > 0).any():
    print('경고(Warning)가 나온 파일:', ', '.join(df.loc[df['경고'] > 0, '파일']))